# NB06 — Forecast Risk Prediction (30-Day Wildfire Risk)

**Objective:** Use the 30-day weather forecast from NB03 and the trained fire detection model from NB05 to predict **future wildfire risk** for every city × date.

### Pipeline
1. Load 30-day weather forecast (`weather_forecast_30d.parquet`)
2. Load best fire model + threshold from NB05
3. Engineer the same features used during training (calendar, city static, lag proxies)
4. Predict fire probability for each city × date
5. Assign risk levels: Low / Medium / High / Extreme
6. Save `wildfire_risk_30d.parquet` for NB07 (dashboard) and NB08 (analysis)

**Input:** `outputs/weather_forecast_30d.parquet`, `models/wildfire/best_fire_model.joblib`  
**Output:** `outputs/wildfire_risk_30d.parquet`

In [ ]:
# ─── Cell 1: Imports ─────────────────────────────────────────────────────
import os, sys, json, warnings
from pathlib import Path

import numpy as np
import pandas as pd
from joblib import load as jl_load

warnings.filterwarnings("ignore")

sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))
from src.config import (ROOT, PROCESSED, OUTPUTS, MODELS_F, REFERENCE,
                         CITIES, FORECAST_30D, RISK_30D, ENG_DAILY, MASTER_DAILY)

print(f"Root: {ROOT}")

## §1 — Load Weather Forecast & Fire Model

In [ ]:
# ─── §1: Load forecast, model, and historical data ───────────────────────

# 1a. Weather forecast
fc = pd.read_parquet(FORECAST_30D)
fc["Date"] = pd.to_datetime(fc["Date"])
print(f"Weather forecast: {fc.shape}")
print(f"  Columns: {list(fc.columns)}")
print(f"  Date range: {fc['Date'].min().date()} → {fc['Date'].max().date()}")
print(f"  Cities: {fc['City'].nunique() if 'City' in fc.columns else 'N/A'}")

# 1b. Fire model
model_path = MODELS_F / "best_fire_model.joblib"
manifest_path = MODELS_F / "model_manifest.json"

if model_path.exists():
    fire_model = jl_load(model_path)
    print(f"\nFire model loaded: {model_path}")
else:
    raise FileNotFoundError(f"No fire model at {model_path}. Run NB05 first.")

if manifest_path.exists():
    with open(manifest_path) as f:
        manifest = json.load(f)
    THRESHOLD = manifest["optimal_threshold"]
    print(f"  Threshold: {THRESHOLD}")
    print(f"  Model: {manifest['model_name']}")
else:
    THRESHOLD = 0.3
    print(f"  Using default threshold: {THRESHOLD}")

# 1c. Feature column list
feat_path = MODELS_F / "feature_columns.json"
if feat_path.exists():
    with open(feat_path) as f:
        FEATURE_COLS = json.load(f)
    print(f"  Features expected: {len(FEATURE_COLS)}")
else:
    FEATURE_COLS = None
    print("  ⚠ No feature_columns.json — will infer from data")

# 1d. Historical data (for lag features and city static data)
hist_path = ENG_DAILY if ENG_DAILY.exists() else MASTER_DAILY
hist = pd.read_parquet(hist_path)
hist["Date"] = pd.to_datetime(hist["Date"])
hist = hist.sort_values(["City", "Date"])
print(f"\nHistorical data: {hist.shape}")

# 1e. Static geography
static_path = REFERENCE / "static_geography.parquet"
if static_path.exists():
    static_df = pd.read_parquet(static_path)
    print(f"Static geography: {static_df.shape}")
else:
    static_df = pd.DataFrame([{"City": c, "Latitude": lat, "Longitude": lon,
                                "Elevation": 0, "Slope": 0, "Trees_pct": 0,
                                "Urban_pct": 0, "Pop_Total": 0}
                               for c, (lat, lon) in CITIES.items()])
    print("Using placeholder static geography")

## §2 — Engineer Features for Forecast Period

We replicate the feature engineering from NB02 on the forecast data. Since we don't have future lags, we use the **last known historical values** as proxy features. This is a realistic production scenario — at prediction time, you only have history + forecast weather.

In [ ]:
# ─── §2: Build feature matrix for forecast dates ─────────────────────────

# Ensure forecast has City column — expand if needed
if "City" not in fc.columns:
    # Broadcast forecast to all cities
    rows = []
    for city in CITIES:
        tmp = fc.copy()
        tmp["City"] = city
        rows.append(tmp)
    fc = pd.concat(rows, ignore_index=True)
    print(f"Broadcast forecast to {len(CITIES)} cities: {fc.shape}")

# Rename forecast columns to match training feature names
# The forecast may have raw target names; map to daily aggregation names
col_map = {}
for c in fc.columns:
    if c in ["City", "Date"]:
        continue
    # If column is already in the expected format, keep it
    if "_mean" in c or "_sum" in c or "_min" in c or "_max" in c:
        continue
    # Map raw names to _mean suffix
    if c + "_mean" not in fc.columns:
        col_map[c] = c + "_mean"

if col_map:
    fc = fc.rename(columns=col_map)
    print(f"Renamed columns: {col_map}")

# For each city, get the last 90 days of history to compute lag/rolling features
forecast_frames = []

for city in sorted(fc["City"].unique()):
    city_fc = fc[fc["City"] == city].sort_values("Date").copy()
    city_hist = hist[hist["City"] == city].sort_values("Date").copy()

    # Take last 90 days of history + forecast
    cutoff = city_fc["Date"].min() - pd.Timedelta(days=90)
    recent_hist = city_hist[city_hist["Date"] >= cutoff].copy()

    # Concatenate history + forecast
    # Use only columns that exist in both
    common_cols = ["City", "Date"] + [c for c in city_fc.columns
                                       if c in recent_hist.columns and c not in ["City", "Date"]]
    combined = pd.concat([recent_hist[common_cols], city_fc[common_cols]],
                          ignore_index=True)
    combined = combined.sort_values("Date").drop_duplicates(subset=["City", "Date"], keep="last")

    # Calendar features
    combined["Year"]       = combined["Date"].dt.year
    combined["Month"]      = combined["Date"].dt.month
    combined["DayOfYear"]  = combined["Date"].dt.dayofyear
    combined["DayOfWeek"]  = combined["Date"].dt.dayofweek
    combined["WeekOfYear"] = combined["Date"].dt.isocalendar().week.astype(int)
    combined["Month_sin"]  = np.sin(2 * np.pi * combined["Month"] / 12)
    combined["Month_cos"]  = np.cos(2 * np.pi * combined["Month"] / 12)
    combined["DoY_sin"]    = np.sin(2 * np.pi * combined["DayOfYear"] / 365)
    combined["DoY_cos"]    = np.cos(2 * np.pi * combined["DayOfYear"] / 365)
    combined["DoW_sin"]    = np.sin(2 * np.pi * combined["DayOfWeek"] / 7)
    combined["DoW_cos"]    = np.cos(2 * np.pi * combined["DayOfWeek"] / 7)
    combined["Season"]     = combined["Month"].map(
        {12:0,1:0,2:0,3:1,4:1,5:1,6:2,7:2,8:2,9:3,10:3,11:3})
    combined["is_summer"]      = combined["Month"].isin([6,7,8]).astype(int)
    combined["is_winter"]      = combined["Month"].isin([12,1,2]).astype(int)
    combined["is_fire_season"] = combined["Month"].isin([5,6,7,8,9]).astype(int)

    # Lag features from history
    lag_vars = [c for c in combined.columns if any(k in c for k in
        ["Temperature_C", "Humidity_percent", "Rain_mm", "Wind_Speed",
         "Pressure", "Solar", "Soil_Temp", "Soil_Moisture", "FWI_proxy"])
        and c not in ["City", "Date"] and combined[c].dtype in ["float64", "float32", "int64"]]

    for var in lag_vars:
        for lag in [1, 2, 3, 5, 7, 14, 30]:
            combined[f"{var}_lag{lag}"] = combined[var].shift(lag)
        for w in [3, 7, 14, 30]:
            shifted = combined[var].shift(1)
            combined[f"{var}_roll{w}_mean"] = shifted.rolling(w, min_periods=1).mean()
            combined[f"{var}_roll{w}_std"]  = shifted.rolling(w, min_periods=1).std()

    # Merge static geography
    combined = combined.merge(static_df, on="City", how="left")

    # Keep only forecast dates
    fc_dates = city_fc["Date"].values
    forecast_only = combined[combined["Date"].isin(fc_dates)].copy()
    forecast_frames.append(forecast_only)

forecast_df = pd.concat(forecast_frames, ignore_index=True)
forecast_df = forecast_df.fillna(0)

print(f"\nForecast feature matrix: {forecast_df.shape}")
print(f"  Date range: {forecast_df['Date'].min().date()} → {forecast_df['Date'].max().date()}")
print(f"  Cities: {forecast_df['City'].nunique()}")

## §3 — Predict Wildfire Risk

In [ ]:
# ─── §3: Predict fire probability for each city × date ───────────────────

# Align columns with what the model expects
if FEATURE_COLS is not None:
    # Add any missing columns as 0
    for col in FEATURE_COLS:
        if col not in forecast_df.columns:
            forecast_df[col] = 0
    X_forecast = forecast_df[FEATURE_COLS].fillna(0)
else:
    # Fallback: use all numeric columns except ID/target
    drop = ["City", "Date", "Timestamp", "Fire_Occurred",
            "fire_count", "mean_brightness", "max_frp", "Burned_Area_hectares"]
    X_forecast = forecast_df[[c for c in forecast_df.columns
                               if c not in drop and forecast_df[c].dtype in
                               ["float64","float32","int64","int32"]]].fillna(0)

print(f"Prediction matrix: {X_forecast.shape}")

# Predict probabilities
fire_proba = fire_model.predict_proba(X_forecast)[:, 1]
fire_pred  = (fire_proba >= THRESHOLD).astype(int)

# Build output
risk_df = forecast_df[["City", "Date"]].copy()

# Add key weather columns if available
for col in ["Temperature_C_mean", "Humidity_percent_mean", "Rain_mm_sum",
            "Wind_Speed_kmh_mean", "Pressure_hPa_mean"]:
    if col in forecast_df.columns:
        risk_df[col] = forecast_df[col].values

risk_df["fire_probability"] = fire_proba
risk_df["fire_predicted"]   = fire_pred

# Risk levels
def risk_level(p):
    if p < 0.15:   return "Low"
    elif p < 0.35: return "Medium"
    elif p < 0.60: return "High"
    else:          return "Extreme"

risk_df["risk_level"] = risk_df["fire_probability"].apply(risk_level)

# Add latitude/longitude for mapping
city_coords = pd.DataFrame([{"City": c, "Latitude": lat, "Longitude": lon}
                             for c, (lat, lon) in CITIES.items()])
risk_df = risk_df.merge(city_coords, on="City", how="left")

# Save
risk_df.to_parquet(RISK_30D, index=False)
risk_df.to_csv(OUTPUTS / "wildfire_risk_30d.csv", index=False)

print(f"\nRisk predictions saved: {risk_df.shape}")
print(f"  File: {RISK_30D}")
print(f"\nRisk distribution:")
print(risk_df["risk_level"].value_counts().to_string())
print(f"\nTop 10 highest-risk city-dates:")
print(risk_df.nlargest(10, "fire_probability")[
    ["City", "Date", "fire_probability", "risk_level"]].to_string(index=False))

# Summary by city
print(f"\nMean fire probability by city:")
city_risk = risk_df.groupby("City")["fire_probability"].agg(["mean", "max"]).round(4)
city_risk = city_risk.sort_values("mean", ascending=False)
print(city_risk.to_string())

print(f"\n→ Next: open 07_Interactive_Map_Dashboard.ipynb")